# Predict week

Builds features for this season, loads the saved models and writes one predictions file for the week to `outputs/`. Run every week.

In [1]:
import sys
sys.path.append('..')

from src import config, features

In [2]:
import pandas as pd
from xgboost import XGBRegressor

YEAR, WEEK = config.CURRENT_YEAR, config.WEEK
print(f'Predicting {YEAR} week {WEEK}')

Predicting 2026 week 5


## 1. Players to predict
The top `config.DEPTH_CHART_WRS` WRs on each team's latest depth chart, with their game this week. Teams on bye are left out.

Games of this week that were already played are still predicted from the earlier weeks only, so you can run this on a Friday or re-run a past week to compare with what happened. The depth chart is always the latest one, though.

In [3]:
upcoming = features.load_upcoming_players(YEAR, WEEK)
games = features.load_week_games(YEAR, WEEK)
print(f'{len(upcoming)} players from {upcoming["ball_team"].nunique()} teams')

all_teams = set(features.load_week_games(YEAR, 1)['ball_team'])
print('On bye:', sorted(all_teams - set(games['ball_team'])) or 'none')
played = games[games['played']]['game_id'].unique()
if len(played):
    print(f'Already played ({len(played)}):', ', '.join(played))

90 players from 30 teams
On bye: ['CAR', 'KC']
Already played (1): 2026_05_TB_DAL


## 2. Build the features
Same function as the training data, loaded from the first training season so the career averages are complete.

In [4]:
years = config.TRAIN_YEARS + [YEAR]
table = features.build_feature_table(years, upcoming)
week_rows = table[(table['season'] == YEAR) & (table['week'] == WEEK)].merge(
    upcoming[['player_id', 'depth_rank']], on='player_id')
print(f'{len(week_rows)} rows to predict')

no_history = week_rows[week_rows[f'receptions_last{config.N_GAMES}'].isna()]
if len(no_history):
    print('No earlier games (predicted from team features only):', ', '.join(no_history['player_display_name']))

90 rows to predict


## 3. Predict
Each saved model uses the features it was trained with (`feature_names_in_`), so later changes to `config.MODEL_FEATURES` don't break older models.

In [5]:
predictions = week_rows[['player_id', 'player_display_name', 'ball_team', 'opponent_team', 'game_id', 'depth_rank']].copy()
for target in config.TARGETS:
    model = XGBRegressor()
    model.load_model(config.model_path(target))
    predictions[f'pred_{target}'] = model.predict(week_rows[model.feature_names_in_])

# Context for reading the predictions
context = ['receptions_season_avg', 'receiving_yards_season_avg', 'receiving_tds_season_avg',
           'target_share_season_avg', f'receiving_yards_last{config.N_GAMES}', 'prev_offense_pct']
predictions = predictions.join(week_rows[context])
predictions = predictions.sort_values('pred_receiving_yards', ascending=False).reset_index(drop=True)
predictions.head(20).round(2)

,player_id,player_display_name,ball_team,opponent_team,game_id,depth_rank,pred_receptions,pred_receiving_yards,pred_receiving_tds,receptions_season_avg,receiving_yards_season_avg,receiving_tds_season_avg,target_share_season_avg,receiving_yards_last3,prev_offense_pct
0,00-0039075,Puka Nacua,LA,BUF,2026_05_BUF_LA,1,6.60,93.300003,0.43,7.00,99.50,0.00,0.29,91.67,0.90
1,00-0036963,Amon-Ra St. Brown,DET,ARI,2026_05_DET_ARI,1,6.60,82.320000,0.59,7.75,75.75,1.25,0.29,78.67,0.96
2,00-0036358,CeeDee Lamb,DAL,TB,2026_05_TB_DAL,1,6.91,81.349998,0.50,9.25,124.50,1.00,0.31,151.33,0.86
3,00-0038543,Jaxon Smith-Njigba,SEA,SF,2026_05_SF_SEA,1,6.41,79.620003,0.52,8.00,120.25,1.50,0.37,119.67,0.88
4,00-0036912,DeVonta Smith,PHI,JAX,2026_05_PHI_JAX,1,5.74,77.099998,0.43,6.33,78.33,0.33,0.33,78.33,1.00
5,00-0037239,Chris Olave,NO,MIN,2026_05_MIN_NO,1,6.17,76.309998,0.46,8.75,122.75,0.25,0.29,103.00,0.86
6,00-0036410,Tee Higgins,CIN,MIA,2026_05_CIN_MIA,2,5.57,74.330002,0.47,6.25,100.25,0.25,0.25,114.00,0.79
7,00-0031381,Davante Adams,LA,BUF,2026_05_BUF_LA,2,5.55,72.849998,0.49,5.50,97.50,0.50,0.25,121.33,0.95
8,00-0039064,Zay Flowers,BAL,ATL,2026_05_BAL_ATL,1,5.27,72.349998,0.42,6.00,117.33,0.67,0.30,117.33,0.73
9,00-0036322,Justin Jefferson,MIN,NO,2026_05_MIN_NO,1,4.88,71.739998,0.39,4.33,59.67,0.67,0.26,59.67,0.12


## 4. Save

In [6]:
path = config.predictions_path(YEAR, WEEK)
path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(path, index=False)
print('Saved', path)

Saved C:\Users\rille\sportsLab_NFLFantasy_projection\Weekly projections\WR\outputs\2026\wrs_predictions_week5.csv
